PySpark Hands-on: MLlib

1. Distinguir **Transformers** de **Estimators** en MLlib.
2. Preparar features con `VectorAssembler`, `StringIndexer`, `OneHotEncoder`, `StandardScaler`.
3. Construir un **Pipeline** end-to-end (preprocesamiento + modelo).
4. Hacer **train/test split** correctamente.
5. Entrenar **regresión lineal** y **regresión logística**.
6. Entrenar **Random Forest** (clasificación).
7. Evaluar modelos con métricas estándar: RMSE, MAE, R² (regresión); AUC, F1, accuracy (clasificación).
8. Hacer **CrossValidator** + grid search para hiperparámetros.
9. Aplicar **K-Means** para clustering no supervisado.
10. Construir un **sistema de recomendación con ALS** usando MovieLens.

> **Dataset:** MovieLens (el mismo de la tarea). Usaremos ratings + movies para:
> - **Bloque 1**: regresión — predecir el rating de una película según features derivadas.
> - **Bloque 2**: clasificación — ¿es una película "bien evaluada" (≥ 4.0)?
> - **Bloque 3**: ALS — recomendar películas a usuarios.


---
# BLOQUE 1 — Fundamentos, Pipelines y Regresión Lineal (60 min)

## 1.1 — Setup


In [ ]:
import os
import sys
os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Mlib en PySpark")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession lista. Versión:", spark.version)
print("SparkUI corre en:", spark.sparkContext.uiWebUrl)

In [ ]:
# Descargar MovieLens
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O /tmp/ml.zip
!unzip -q -o /tmp/ml.zip -d /tmp/
!ls /tmp/ml-latest-small/

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, LongType
from pyspark.sql.functions import col

schema_movies = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title",   StringType(),  True),
    StructField("genres",  StringType(),  True),
])
schema_ratings = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("rating",    DoubleType(),  True),
    StructField("timestamp", LongType(),    True),
])

df_movies = spark.read.schema(schema_movies).option("header", "true").csv("/tmp/ml-latest-small/movies.csv")
df_ratings = spark.read.schema(schema_ratings).option("header", "true").csv("/tmp/ml-latest-small/ratings.csv")

print(f"Películas: {df_movies.count():,}")
print(f"Ratings:   {df_ratings.count():,}")

Ahora, vamos a mostrar las primeras filas de ambos DataFrames para verificar que se cargaron correctamente.

In [ ]:
print('Primeras 5 filas de df_movies:')
df_movies.show(5, truncate=False)

In [ ]:
print('Primeras 5 filas de df_ratings:')
df_ratings.show(5, truncate=False)

## 1.2 — MLlib: dos conceptos clave

MLlib tiene una API muy elegante (inspirada en scikit-learn) basada en dos abstracciones:

### Transformer
**Una caja que recibe un DataFrame y devuelve otro DataFrame.** No aprende nada. Solo transforma.

Ejemplos: `VectorAssembler`, `StandardScaler` (después de entrenado), `Tokenizer`.

Tienen un método `.transform(df) -> df_nuevo`.

### Estimator
**Una caja que aprende algo de un DataFrame y devuelve un Transformer (su modelo entrenado).**

Ejemplos: `LinearRegression`, `RandomForestClassifier`, `StringIndexer`, `KMeans`.

Tienen un método `.fit(df) -> Modelo` (que es un Transformer).

```
LinearRegression  (Estimator)    .fit(train_df)   →  LinearRegressionModel  (Transformer)
                                                       .transform(test_df)  →  predicciones_df
```

### Pipeline
Une varios Transformers + Estimators en una secuencia. Se entrena todo junto.

```
Pipeline([VectorAssembler, StandardScaler, LinearRegression])
    .fit(train) → PipelineModel
    .transform(test) → predicciones
```

Vamos a verlo con un caso concreto.


## 1.3 — Feature engineering: preparar las features

Vamos a predecir el **rating promedio de una película** a partir de:
- Cantidad de ratings que tiene (popularidad).
- Año de la película.
- Géneros (codificados).

Primero, construimos el dataset de features.


In [ ]:
from pyspark.sql.functions import (
    avg, count, year, regexp_extract, regexp_replace,
    split, lit, when, col
)

# Stats por película
df_stats = (
    df_ratings.groupBy("movieId")
    .agg(
        count("*").alias("n_ratings"),
        avg("rating").alias("rating_promedio")
    )
    .filter(col("n_ratings") >= 30)   # solo películas con suficientes ratings
)

# Año y limpieza de título.
# OJO: en Spark 4 (ANSI mode ON), cast("") a int falla.
# Protegemos con when(): solo casteamos si el regex encontró algo.
"""
| Parte     | Significado                           |
| --------- | ------------------------------------- |
| `\(`      | busca un paréntesis de apertura `(`   |
| `(\d{4})` | captura 4 dígitos, por ejemplo `1995` |
| `\)`      | busca un paréntesis de cierre `)`     |
| `$`       | exige que eso esté al final del texto |

"""

df_mov = (
    df_movies
    .withColumn(
        "anio",
        when(
            regexp_extract(col("title"), r"\((\d{4})\)$", 1) != "",
            regexp_extract(col("title"), r"\((\d{4})\)$", 1).cast("int")
        )
    )
    .withColumn("titulo_limpio", regexp_replace(col("title"), r"\s*\(\d{4}\)$", ""))
    .filter(col("anio").isNotNull())
)

# Unimos
df_feat = (
    df_stats.join(df_mov, on="movieId", how="inner")
    .select("movieId", "titulo_limpio", "anio", "genres", "n_ratings", "rating_promedio")
)

df_feat.show(5, truncate=False)
print(f"Filas para entrenar: {df_feat.count():,}")

## 1.4 — Preprocesamiento: features numéricas y categóricas

Spark MLlib espera **un único vector de features por fila** (columna llamada `features`). Para llegar ahí:




1. Features numéricas se juntan con `VectorAssembler`.

| anio | n_ratings | features      |
| ---: | --------: | ------------- |
| 1995 |       215 | `[1995, 215]` |
| 2001 |        87 | `[2001, 87]`  |
| 2010 |       430 | `[2010, 430]` |

2. Features categóricas (strings) se convierten a numéricas con `StringIndexer` + opcionalmente `OneHotEncoder`.


| genero | genero_indexado |
| ------ | --------------: |
| Comedy |               0 |
| Drama  |               1 |
| Action |               2 |


| genero | one_hot     |
| ------ | ----------- |
| Comedy | `[1, 0, 0]` |
| Drama  | `[0, 1, 0]` |
| Action | `[0, 0, 1]` |


3. Features numéricas de escalas distintas se escalan con `StandardScaler`.



Vamos a usar como features: `anio` y `n_ratings` (numéricas). Como **target** (lo que queremos predecir): `rating_promedio`.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1) Juntar features numéricas en un vector
assembler = VectorAssembler(
    inputCols=["anio", "n_ratings"],
    outputCol="features_raw"
)

# 2) Escalar (media 0, desviación 1) — importante para muchos modelos
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Probarlo paso a paso
df_assembled = assembler.transform(df_feat)
df_assembled.select("features_raw", "rating_promedio").show(5, truncate=False)

#### Después de `VectorAssembler` (sin escalar)

La columna `features_raw` ahora contiene un vector con los valores de `anio` y `n_ratings` para cada película. Esto se puede ver en la salida `df_assembled.select("features_raw", "rating_promedio").show(5, truncate=False)` justo arriba.

```
+--------------+------------------+
|features_raw  |rating_promedio   |
+--------------+------------------+
|[1996.0,55.0] |3.5090909090909093|
|[1994.0,34.0] |3.5294117647058822|
|[1993.0,190.0]|3.9921052631578946|
|[1996.0,53.0] |2.707547169811321 |
|[1941.0,35.0] |3.3857142857142857|
+--------------+------------------+
```

In [ ]:
# Notar que scaler es un ESTIMATOR (tiene que aprender la media y std)
scaler_model = scaler.fit(df_assembled)            # ahora es un Transformer entrenado
df_scaled = scaler_model.transform(df_assembled)
df_scaled.select("features_raw", "features", "rating_promedio").show(5, truncate=False)

#### Después de `StandardScaler`

La columna `features` ahora contiene los mismos valores, pero escalados para tener media 0 y desviación estándar 1. Esto es crucial para que los modelos de regresión lineal no den más peso a features con rangos de valores más grandes.

```
+--------------+------------------------------------------+------------------+
|features_raw  |features                                  |rating_promedio   |
+--------------+------------------------------------------+------------------+
|[1996.0,55.0] |[0.10782882407579078,-0.25338067796198793]|3.5090909090909093|
|[1994.0,34.0] |[-0.04421757534050332,-0.7469773247028343]|3.5294117647058822|
|[1993.0,190.0]|[-0.12024077504865038,2.919740622514882]  |3.9921052631578946|
|[1996.0,53.0] |[0.10782882407579078,-0.30038988241349707]|2.707547169811321 |
|[1941.0,35.0] |[-4.073447159872297,-0.7234727224770797]  |3.3857142857142857|
+--------------+------------------------------------------+------------------+
```

## 1.5 — Train/test split

Antes de entrenar, partimos los datos. Estándar: 80/20.

**Importante:** usar `seed` fijo para que la partición sea reproducible.


In [ ]:
train, test = df_feat.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,} filas")
print(f"Test:  {test.count():,} filas")

## 1.6 — Pipeline: encadenar todo

En lugar de aplicar cada paso a mano, los metemos en un `Pipeline`. Ventaja: **entrenas el pipeline completo y se aplica el mismo preprocesamiento a train y test**.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression

# Construimos el pipeline
lr = LinearRegression(
    featuresCol="features",
    labelCol="rating_promedio",
    predictionCol="prediccion"
)

pipeline = Pipeline(stages=[assembler, scaler, lr])

# Entrenar
modelo = pipeline.fit(train)
print("Modelo entrenado")

#### Estructura del `Pipeline` (Modelo sin género)

El `pipeline` encapsula los siguientes pasos, que se aplicarán secuencialmente a los datos:

1.  **`VectorAssembler`**: Toma las columnas `anio` y `n_ratings` y las combina en una sola columna de tipo vector llamada `features_raw`.
2.  **`StandardScaler`**: Escala la columna `features_raw` para que tenga media 0 y desviación estándar 1, generando la columna `features`.
3.  **`LinearRegression`**: Utiliza la columna `features` como entrada para el modelo y predice el `rating_promedio`.

In [ ]:
# Predecir sobre test
predicciones = modelo.transform(test)
predicciones.select("titulo_limpio", "rating_promedio", "prediccion").show(10, truncate=False)

## 1.7 — Evaluar modelos de regresión

Métricas para regresión:

| Métrica | Qué mide |
|---------|----------|
| **RMSE** | Error cuadrático medio (raíz). Castiga errores grandes. | También mide cuánto se equivoca el modelo,
| **MAE** | Error absoluto medio. Más intuitivo. | En promedio, ¿cuánto se equivoca el modelo?
| **R²** | Proporción de varianza explicada. 1.0 = perfecto, 0.0 = igual que predecir el promedio. |

Si RMSE es mucho mayor que MAE, puede significar que el modelo tiene algunos errores muy grandes.

R² => ¿Mi modelo realmente explica algo o solo predice el promedio?


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="rating_promedio",
    predictionCol="prediccion"
)

rmse = evaluator.setMetricName("rmse").evaluate(predicciones)
mae  = evaluator.setMetricName("mae").evaluate(predicciones)
r2   = evaluator.setMetricName("r2").evaluate(predicciones)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")



**¿Qué nos dice esto?**

Con solo `anio` y `n_ratings`, R² suele dar bajo (0.10 a 0.20). Tiene sentido: el rating depende mucho más del contenido que del año o la popularidad.

**Mejorar el modelo** = agregar mejores features. Por ejemplo: género, director, actores. En la sección siguiente metemos género usando StringIndexer + OneHotEncoder.


### Guardar y cargar modelos

Es fundamental poder guardar los modelos entrenados para poder reutilizarlos sin tener que re-entrenar, o para desplegarlos en producción. Los modelos de MLlib (que en realidad son `PipelineModel` si se usa un `Pipeline`) tienen métodos `save()` y `load()`.

In [ ]:
# Guardar el primer modelo de regresión lineal
modelo.write().overwrite().save("/tmp/linear_regression_model")
print("Modelo de Regresión Lineal (sin género) guardado en /tmp/linear_regression_model")

También podemos cargar un modelo previamente guardado.

In [ ]:
from pyspark.ml import PipelineModel

# Cargar el modelo guardado
loaded_model = PipelineModel.load("/tmp/linear_regression_model")
print("Modelo cargado exitosamente.")

# Puedes usar el modelo cargado para hacer predicciones
# loaded_predictions = loaded_model.transform(test)
# loaded_predictions.select("titulo_limpio", "rating_promedio", "prediccion").show(5, truncate=False)

## 1.8 — Agregar géneros como feature

`genres` es un string como `"Comedy|Romance"`. Hay dos enfoques:

1. **OneHotEncoder por género**: una columna por género. Muchas columnas, pero captura todo.
2. **Tomar solo el primer género**: simple, pero pierde info.

Vamos con la opción simple para esta demo (primer género) y dejamos OneHot como ejercicio.


#### Extracción del `genero_principal`

Aquí se crea una nueva columna `genero_principal` extrayendo el primer género de la columna `genres`. Por ejemplo:

| genres                                            | genero_principal |
| :------------------------------------------------ | :--------------- |
| Adventure|Animation|Children|Comedy|Fantasy | Adventure        |
| Adventure|Children|Fantasy                       | Adventure        |
| Comedy|Romance                                   | Comedy           |
| Comedy|Drama|Romance                              | Comedy           |
| Comedy                                            | Comedy           |

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import split

df_feat2 = df_feat.withColumn("genero_principal", split(col("genres"), r"\|").getItem(0))

# StringIndexer: género string -> índice numérico
indexer = StringIndexer(
    inputCol="genero_principal",
    outputCol="genero_idx",
    handleInvalid="keep"
)

# OneHotEncoder: índice -> vector sparse
ohe = OneHotEncoder(
    inputCols=["genero_idx"],
    outputCols=["genero_vec"]
)

# Nuevo assembler con la feature de género
assembler2 = VectorAssembler(
    inputCols=["anio", "n_ratings", "genero_vec"],
    outputCol="features_raw"
)

train2, test2 = df_feat2.randomSplit([0.8, 0.2], seed=42)

pipeline2 = Pipeline(stages=[indexer, ohe, assembler2, scaler, lr])
modelo2 = pipeline2.fit(train2)

predicciones2 = modelo2.transform(test2)
rmse2 = evaluator.setMetricName("rmse").evaluate(predicciones2)
r2_2  = evaluator.setMetricName("r2").evaluate(predicciones2)
print(f"Modelo con género  → RMSE: {rmse2:.4f}, R²: {r2_2:.4f}")
print(f"Modelo sin género  → RMSE: {rmse:.4f}, R²: {r2:.4f}")

#### Después de `StringIndexer` y `OneHotEncoder`

Cuando se incluye `genero_principal` en el pipeline, primero `StringIndexer` lo convierte en un índice numérico (`genero_idx`), y luego `OneHotEncoder` lo transforma en un vector binario (`genero_vec`).

| genero_principal | genero_idx | genero_vec |
| :--------------- | :--------- | :--------- |
| Comedy           | 0.0        | `[1.0,0.0,0.0,...]` |
| Adventure        | 1.0        | `[0.0,1.0,0.0,...]` |
| Drama            | 2.0        | `[0.0,0.0,1.0,...]` |


El `assembler2` luego combina `anio`, `n_ratings` y este `genero_vec` en una nueva columna `features_raw` que se escalará. De esta forma, el modelo puede usar la información del género.

Ya con el género el R² sube algo.



---
# BLOQUE 2 — Clasificación: Regresión Logística y Random Forest (60 min)

## 2.1 — Planteamiento

Cambiamos de problema: ahora queremos predecir si una película va a ser **"bien evaluada"** (rating ≥ 4.0) o no. Es una **clasificación binaria**.

Etiqueta: `bien_evaluada = 1` si `rating_promedio >= 4.0`, sino `0`.


In [ ]:
df_clas = (
    df_feat2
    .withColumn("label", when(col("rating_promedio") >= 4.0, 1.0).otherwise(0.0))
    .select("movieId", "titulo_limpio", "anio", "n_ratings", "genero_principal", "label")
)

# Distribución de clases
df_clas.groupBy("label").count().show()

#### Interpretación de la Distribución de Clases

La salida de `df_clas.groupBy("label").count().show()` nos muestra la cantidad de películas en cada una de las dos clases que hemos definido para el problema de clasificación:

*   **`label = 1.0`**: Representa las películas consideradas "bien evaluadas" (es decir, aquellas con un `rating_promedio` de 4.0 o superior).
*   **`label = 0.0`**: Representa las películas que **no** son "bien evaluadas" (con un `rating_promedio` inferior a 4.0).

En tu caso:

```
+-----+
|label|count|
+-----+
|  1.0|  124|
|  0.0|  758|
+-----+
```

Esto significa que hay **124 películas** que están en la clase "bien evaluada" y **758 películas** en la clase "no bien evaluada".

Esta distribución revela un **desbalance significativo de clases**. La clase `0.0` es mayoritaria (hay muchas más películas no bien evaluadas que bien evaluadas). Este desbalance es crucial porque:

*   **Métricas engañosas**: Si un modelo siempre predice la clase mayoritaria (0.0 en este caso), podría obtener una alta `accuracy` (precisión general), pero sería completamente inútil para identificar las películas "bien evaluadas".
*   **Rendimiento del modelo**: Los modelos tienden a aprender mejor de la clase con más ejemplos. Esto puede llevar a que un modelo tenga dificultades para clasificar correctamente la clase minoritaria (1.0).

Por estas razones, en problemas con desbalance de clases, es más apropiado evaluar el modelo con métricas como **AUC (Area Under the Receiver Operating Characteristic curve)** o **F1-score**, que son más robustas a este tipo de situaciones que la simple `accuracy`.

Si las clases están desbalanceadas (mucho más 0 que 1, o al revés), las métricas como **accuracy** mienten. Mejor usar **AUC** o **F1**.

## 2.2 — Regresión Logística

El clasificador más simple. Pero subestimado: en muchos problemas reales es competitivo y siempre interpretable.


In [ ]:
from pyspark.ml.classification import LogisticRegression

train3, test3 = df_clas.randomSplit([0.8, 0.2], seed=42)

lr_clas = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediccion",
    probabilityCol="prob",
    maxIter=20
)

assembler3 = VectorAssembler(
    inputCols=["anio", "n_ratings", "genero_vec"],
    outputCol="features_raw"
)

pipeline3 = Pipeline(stages=[indexer, ohe, assembler3, scaler, lr_clas])
modelo_lr = pipeline3.fit(train3)
pred_lr = modelo_lr.transform(test3)

pred_lr.select("titulo_limpio", "label", "prediccion", "prob").show(10, truncate=False)

#### Interpretación de las Predicciones de Regresión Logística

En la celda anterior, se entrenó un modelo de **Regresión Logística** para clasificar si una película es "bien evaluada" (`label = 1.0`) o no (`label = 0.0`).

El resultado `pred_lr.select("titulo_limpio", "label", "prediccion", "prob").show(10, truncate=False)` muestra:

*   **`titulo_limpio`**: El título de la película.
*   **`label`**: La etiqueta real de la película (1.0 si es bien evaluada, 0.0 si no).
*   **`prediccion`**: La predicción del modelo de Regresión Logística para la etiqueta (0.0 o 1.0).
*   **`prob` (probabilityCol)**: Este es un vector que contiene las probabilidades predichas por el modelo para cada clase. El primer valor es la probabilidad de que la película pertenezca a la clase 0.0 (no bien evaluada), y el segundo valor es la probabilidad de que pertenezca a la clase 1.0 (bien evaluada).

Por ejemplo, si `prob` es `[0.91, 0.09]`, significa que el modelo predice un 91% de probabilidad de ser de la clase 0.0 y un 9% de probabilidad de ser de la clase 1.0. En este caso, la `prediccion` sería 0.0.

## 2.3 — Métricas de clasificación

| Métrica | Cuándo usar |
|---------|-------------|
| **Accuracy** | Solo si las clases están balanceadas. |
| **AUC** (Area Under ROC) | Robusto a desbalance. El estándar para clasificación binaria. |
| **F1 score** | Balance entre precisión y recall. |
| **Precision / Recall** | Cuando una clase es más importante (fraude, médico). |


#### Explicación Detallada de las Métricas de Clasificación

Las métricas de clasificación nos ayudan a entender qué tan bien está funcionando un modelo predictivo, especialmente cuando predice categorías (como 'bien evaluada' o 'no bien evaluada').

Vamos a usar un ejemplo sencillo. Imagina que tienes 10 películas y tu modelo de clasificación predice si son 'buenas' (1) o 'malas' (0):

| Película | Etiqueta Real | Predicción del Modelo |
| :------- | :------------ | :-------------------- |
| A        | 1 (Buena)     | 1 (Buena)             |
| B        | 1 (Buena)     | 0 (Mala)              |
| C        | 1 (Buena)     | 1 (Buena)             |
| D        | 0 (Mala)      | 0 (Mala)              |
| E        | 0 (Mala)      | 1 (Buena)             |
| F        | 0 (Mala)      | 0 (Mala)              |
| G        | 0 (Mala)      | 0 (Mala)              |
| H        | 0 (Mala)      | 0 (Mala)              |
| I        | 0 (Mala)      | 0 (Mala)              |
| J        | 0 (Mala)      | 0 (Mala)              |

En este ejemplo tenemos 3 películas 'buenas' y 7 películas 'malas'.

*   **Verdaderos Positivos (VP)**: El modelo predijo 1 y la real es 1. (Películas A, C = 2)
*   **Falsos Positivos (FP)**: El modelo predijo 1 y la real es 0. (Película E = 1)
*   **Verdaderos Negativos (VN)**: El modelo predijo 0 y la real es 0. (Películas D, F, G, H, I, J = 6)
*   **Falsos Negativos (FN)**: El modelo predijo 0 y la real es 1. (Película B = 1)

---

1.  **Accuracy (Exactitud)**
    *   **Qué mide**: La proporción de predicciones correctas sobre el total de predicciones.
    *   **Fórmula**: `(VP + VN) / (VP + VN + FP + FN)`
    *   **Cuándo usar**: Solo si las clases están balanceadas (similar número de ejemplos en cada clase).
    *   **Ejemplo**: `(2 + 6) / (2 + 6 + 1 + 1) = 8 / 10 = 0.8` (80% de las predicciones fueron correctas).
    *   **Por qué puede engañar**: Si el 90% de las películas fueran 'malas' y el modelo siempre predice 'mala', obtendría un 90% de accuracy, pero no sería útil para encontrar las 'buenas'.

2.  **AUC (Area Under the Receiver Operating Characteristic Curve)**
    *   **Qué mide**: La capacidad del modelo para distinguir entre clases. Un valor de 0.5 es aleatorio, 1.0 es perfecto.
    *   **Cuándo usar**: Es la métrica estándar y más **robusta a desbalance de clases** en clasificación binaria.
    *   **No se calcula fácilmente a mano** como las otras, ya que requiere analizar la curva ROC a través de diferentes umbrales de probabilidad.

3.  **F1-score**
    *   **Qué mide**: Es la media armónica de la Precisión y el Recall. Proporciona un balance entre ambas.
    *   **Fórmula**: `2 * (Precision * Recall) / (Precision + Recall)`
    *   **Cuándo usar**: Cuando te importan tanto los Falsos Positivos como los Falsos Negativos, y las clases pueden estar desbalanceadas.
    *   **Calcularemos Primero Precision y Recall para el Ejemplo:**

4.  **Precision (Precisión)**
    *   **Qué mide**: De todas las veces que el modelo predijo positivo, ¿cuántas fueron realmente positivas?
    *   **Fórmula**: `VP / (VP + FP)`
    *   **Cuándo usar**: Cuando el costo de un Falso Positivo es alto (ej. clasificar un correo como spam cuando no lo es).
    *   **Ejemplo**: `2 / (2 + 1) = 2 / 3 = 0.67` (De las veces que predijo 'buena', acertó el 67%).

5.  **Recall (Sensibilidad o Exhaustividad)**
    *   **Qué mide**: De todas las instancias positivas reales, ¿cuántas pudo identificar el modelo?
    *   **Fórmula**: `VP / (VP + FN)`
    *   **Cuándo usar**: Cuando el costo de un Falso Negativo es alto (ej. no detectar una enfermedad grave).
    *   **Ejemplo**: `2 / (2 + 1) = 2 / 3 = 0.67` (De todas las películas 'buenas', identificó el 67%).

    *   **Volviendo al F1-score del Ejemplo**: `2 * (0.67 * 0.67) / (0.67 + 0.67) = 0.67`

#### Columnas Generadas por el Modelo de Clasificación

Cuando se ejecuta un modelo de clasificación como la Regresión Logística, además de la predicción final (`prediccion`), se generan otras columnas que son muy útiles para entender el comportamiento del modelo:

*   **`rawPrediction`**: Esta columna es un vector que contiene los "scores" crudos de la función de decisión del modelo antes de ser transformados en probabilidades o la etiqueta final. En el caso de la regresión logística, estos scores suelen ser los log-odds para cada clase. La clase con el score más alto es la que se predice. Los evaluadores de clasificación como `BinaryClassificationEvaluator` (`areaUnderROC`) a menudo utilizan `rawPrediction` para sus cálculos.

*   **`prob` (probabilityCol)**: Como ya se explicó, este vector contiene las probabilidades predichas por el modelo para cada clase. Por ejemplo, `[prob_clase_0, prob_clase_1]`. Estas probabilidades se obtienen aplicando una función (como la función sigmoide para regresión logística) a `rawPrediction`.

*   **`prediccion` (predictionCol)**: Esta es la etiqueta de clase final predicha por el modelo (0.0 o 1.0 en este caso). Se obtiene seleccionando la clase con la probabilidad más alta de la columna `prob`.

Estas columnas permiten un análisis más profundo del modelo, más allá de solo la etiqueta predicha, y son fundamentales para calcular métricas como el AUC.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC
eval_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc_lr = eval_auc.evaluate(pred_lr)

# F1 y accuracy
eval_multi = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediccion"
)
f1_lr  = eval_multi.setMetricName("f1").evaluate(pred_lr)
acc_lr = eval_multi.setMetricName("accuracy").evaluate(pred_lr)

print(f"Logística → AUC: {auc_lr:.4f}, F1: {f1_lr:.4f}, Accuracy: {acc_lr:.4f}")

#### Interpretación de las Métricas de Clasificación para Regresión Logística

Los resultados obtenidos de la celda anterior para el modelo de Regresión Logística son:

*   **`AUC: 0.8147`**:
    *   **Área bajo la curva ROC**. Un valor de 0.5 indica un modelo que predice al azar, mientras que 1.0 es una predicción perfecta. Un AUC de `0.8147` sugiere que el modelo tiene una **buena capacidad para distinguir entre las películas "bien evaluadas" y las "no bien evaluadas"**.
    *   Es una métrica robusta ante el desbalance de clases, lo que la hace muy adecuada para este problema donde tenemos muchas más películas no bien evaluadas.

*   **`F1: 0.8757`**:
    *   El **F1-score** es la media armónica de la precisión y el recall. Es útil cuando te importan tanto los falsos positivos como los falsos negativos, y cuando hay desbalance de clases.
    *   Un F1 de `0.8757` indica un **buen equilibrio entre la capacidad del modelo para no generar falsos positivos (precisión) y para encontrar todos los positivos reales (recall)**.

*   **`Accuracy: 0.9085`**:
    *   La **exactitud** o `accuracy` representa la proporción de predicciones correctas sobre el total. Un `accuracy` de `0.9085` parece muy bueno a primera vista.
    *   Sin embargo, debido al desbalance de clases que observamos (muchas más películas no bien evaluadas), esta métrica **puede ser engañosa**. Si el modelo predijera '0' para la mayoría de las películas, su `accuracy` sería alta incluso si fallara en identificar casi todas las películas '1'. Por eso, el AUC y el F1-score son más confiables en este contexto.

## 2.4 — Random Forest

Modelo basado en árboles. Suele dar mejor accuracy que regresión logística "out-of-the-box", y casi no requiere preprocesamiento (no necesita escalar).


### Explicación Detallada de Random Forest

**Random Forest** es un algoritmo de aprendizaje automático versátil y potente que se utiliza tanto para tareas de clasificación como de regresión. Pertenece a la familia de los **modelos de ensamble (ensemble learning)**, lo que significa que combina las predicciones de múltiples modelos base para mejorar la precisión y reducir el sobreajuste (overfitting).

En el corazón de Random Forest están los **árboles de decisión**. Un solo árbol de decisión puede ser propenso al sobreajuste si se le permite crecer demasiado profundo, capturando ruido en los datos de entrenamiento.

Random Forest aborda esto de dos maneras clave:

1.  **"Random" en la selección de datos (Bagging)**:
    *   En lugar de entrenar un solo árbol con todo el conjunto de datos, Random Forest crea múltiples árboles de decisión. Cada árbol se entrena con una **muestra aleatoria (bootstrap sample)** del conjunto de datos original. Esto significa que cada árbol ve una versión ligeramente diferente de los datos de entrenamiento.

2.  **"Random" en la selección de features**:
    *   Cuando cada árbol de decisión se construye, en cada paso de división (cuando decide qué característica usar para dividir los datos), no considera todas las características disponibles. En cambio, selecciona un **subconjunto aleatorio de características**. Esto obliga a los árboles a ser más diversos y evita que un solo árbol de decisión domine el modelo si una característica es muy fuerte.

### Cómo funciona el "Bosque" para hacer una predicción:

*   **Para Clasificación**: Cada árbol en el bosque "vota" por una clase, y el modelo de Random Forest predice la clase que recibe la mayoría de los votos (lo que se conoce como "votación mayoritaria").
*   **Para Regresión**: Cada árbol predice un valor, y el modelo de Random Forest toma el promedio de todas esas predicciones.

### Ventajas de Random Forest:

*   **Alta Precisión**: Generalmente ofrece una muy buena precisión, a menudo superando a modelos más simples como la regresión logística.
*   **Menos Sobreajuste**: La aleatoriedad en la construcción de los árboles ayuda a reducir el sobreajuste.
*   **Manejo de Datos Faltantes y Valores Atípicos**: Es relativamente robusto a los datos ruidosos y a los valores atípicos.
*   **Manejo de Diferentes Tipos de Datos**: Puede trabajar con características numéricas y categóricas.
*   **Importancia de las Características (Feature Importance)**: Puede proporcionar una estimación de qué características son más importantes para la predicción, lo que ayuda a la interpretabilidad.
*   **No Requiere Escalado de Características**: A diferencia de modelos basados en distancia (como SVM o K-Means) o modelos lineales (como la regresión logística y lineal), los árboles de decisión (y por extensión, Random Forest) no se ven afectados por la escala de las características. Esto simplifica el preprocesamiento de datos, eliminando la necesidad de `StandardScaler` *para las características del árbol en sí*. Sin embargo, si el pipeline incluye otros modelos o etapas que sí lo requieren, el escalado aún puede ser parte del proceso general, como en este notebook donde `scaler` se aplica antes de `rf`.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediccion",
    numTrees=50,
    maxDepth=8,
    seed=42
)

# Pipeline (mismo pre-procesamiento, distinto modelo)
pipeline_rf = Pipeline(stages=[indexer, ohe, assembler3, scaler, rf])
modelo_rf = pipeline_rf.fit(train3)
pred_rf = modelo_rf.transform(test3)

auc_rf = eval_auc.evaluate(pred_rf)
f1_rf  = eval_multi.setMetricName("f1").evaluate(pred_rf)
acc_rf = eval_multi.setMetricName("accuracy").evaluate(pred_rf)

print(f"Random Forest → AUC: {auc_rf:.4f}, F1: {f1_rf:.4f}, Accuracy: {acc_rf:.4f}")
print(f"Logística     → AUC: {auc_lr:.4f}, F1: {f1_lr:.4f}, Accuracy: {acc_lr:.4f}")

**Feature importance del Random Forest:** podemos ver qué features pesan más en las decisiones del modelo.

In [ ]:
# Acceder al modelo dentro del Pipeline (último stage)
rf_model = modelo_rf.stages[-1]
print("Feature importance:")
print(rf_model.featureImportances)

#### Interpretación de Feature Importance

El output de `rf_model.featureImportances` es un vector disperso (`SparseVector`) que indica la importancia relativa de cada característica (`feature`) que el modelo Random Forest utilizó para hacer sus predicciones. Se muestra en el formato `(size, indices, values)`:

*   **`size` (17)**: El número total de características que el modelo podría haber utilizado. Esto incluye `anio`, `n_ratings` y cada una de las categorías expandidas de `genero_vec` (generadas por el `OneHotEncoder`).
*   **`indices`**: La lista de los índices de las características que tienen una importancia no nula. Es decir, las características que el modelo consideró relevantes.
*   **`values`**: La importancia de cada característica correspondiente a los `indices`. Estos valores son proporcionales y suman 1. Cuanto mayor es el valor, más influyente fue esa característica en las decisiones de los árboles dentro del Random Forest.

Para interpretar esto, necesitamos recordar cómo se construyó la columna `features` en el `VectorAssembler` (`assembler3`):

*   El primer componente (índice 0) corresponde a `anio`.
*   El segundo componente (índice 1) corresponde a `n_ratings`.
*   Los componentes restantes (índices 2 en adelante) corresponden a las diferentes categorías de género después de ser One-Hot Encoded (`genero_vec`).

Analizando los valores de importancia que se imprimieron:

*   **Índice 0 (correspondiente a `anio`)**: Tiene una importancia de aproximadamente `0.3448`.
*   **Índice 1 (correspondiente a `n_ratings`)**: Tiene una importancia de aproximadamente `0.4055`.
*   **Índices 2 en adelante (correspondientes a los géneros)**: Tienen valores de importancia menores, siendo el más alto alrededor de `0.067` para un género específico y otros muy bajos (incluso cercanos a cero como `1.7e-05`).

**Conclusión:**

De esta manera, podemos observar que las características `n_ratings` y `anio` son las más influyentes para el modelo Random Forest en la predicción de si una película está bien evaluada. Individualmente, los géneros tienen una influencia mucho menor en comparación, aunque su conjunto sí contribuye a la capacidad predictiva del modelo.

## 2.5 — CrossValidator: tunear hiperparámetros

Hasta acá usamos hiperparámetros "a mano" (`numTrees=50`, `maxDepth=8`). Para encontrar los mejores, usamos `CrossValidator` con un grid:


In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Grid de hiperparámetros a probar
grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 80])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

cv = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=grid,
    evaluator=eval_auc,
    numFolds=3,
    seed=42,
    parallelism=2
)

# Entrenar — esto demora más (3 folds × 4 combos = 12 entrenamientos)
print("Entrenando CrossValidator (puede tardar 1-3 min)...")
cv_modelo = cv.fit(train3)

print("Mejor AUC en CV:", max(cv_modelo.avgMetrics))
print("Mejores parámetros:")
best_rf = cv_modelo.bestModel.stages[-1]
print(f"  numTrees: {best_rf.getNumTrees}")
print(f"  maxDepth: {best_rf.getOrDefault('maxDepth')}")


#### Interpretación de los Resultados:

*   **`Entrenando CrossValidator (puede tardar 1-3 min)...`**: Este mensaje es un indicio de que el proceso es intensivo computacionalmente, ya que implica múltiples entrenamientos y evaluaciones (en este caso, 12 entrenamientos en total).

*   **`Mejor AUC en CV: 0.7689872085341097`**:
    *   Este es el **valor de AUC promedio más alto** que se obtuvo en el proceso de validación cruzada entre todas las combinaciones de hiperparámetros probadas. Este valor es una estimación del rendimiento del modelo "generalizado" con los mejores parámetros encontrados.

*   **`Mejores parámetros:`**
    *   **`numTrees: 80`**: El `CrossValidator` determinó que, entre los valores `[30, 80]` probados, usar `80` árboles dio el mejor rendimiento (mayor AUC promedio).
    *   **`maxDepth: 10`**: Similarmente, entre las profundidades `[5, 10]` probadas, una `maxDepth` de `10` resultó en el mejor AUC promedio.

**En resumen:** `CrossValidator` te dice que, para este problema y con los datos de entrenamiento, un `RandomForestClassifier` con 80 árboles y una profundidad máxima de 10 es la mejor configuración de hiperparámetros para maximizar el AUC, según la validación cruzada de 3 pliegues.

In [ ]:
# Evaluar el mejor modelo en test
pred_best = cv_modelo.transform(test3)
auc_best = eval_auc.evaluate(pred_best)
print(f"AUC del mejor modelo en test: {auc_best:.4f}")

---
# BLOQUE 3 — K-Means, ALS y sistema de recomendación (60 min)

## 3.1 — Aprendizaje no supervisado: K-Means

Hasta acá hicimos **aprendizaje supervisado**: teníamos un `label` (lo que queríamos predecir) y entrenamos para acercarnos a él.

En **no supervisado** no hay label. Solo queremos **encontrar grupos naturales** en los datos.

**K-Means** es el algoritmo más usado. Le dices cuántos grupos (K) querés, y él los encuentra.

Vamos a agrupar las películas en K grupos según `anio` y `n_ratings`. ¿Vamos a ver patrones tipo "películas viejas con pocos ratings" o "películas recientes muy populares"?


In [ ]:
from pyspark.ml.clustering import KMeans

# Mismo pipeline pero sin escalar la etiqueta (no hay etiqueta)
assembler_km = VectorAssembler(
    inputCols=["anio", "n_ratings"],
    outputCol="features_raw"
)

# Para clustering es FUNDAMENTAL escalar (sino el feature con escala grande domina)
pipeline_pre = Pipeline(stages=[assembler_km, scaler])
df_para_km = pipeline_pre.fit(df_feat).transform(df_feat)

kmeans = KMeans(featuresCol="features", k=4, seed=42)
km_model = kmeans.fit(df_para_km)

# Asignar cluster
df_clustered = km_model.transform(df_para_km)
df_clustered.groupBy("prediction").count().orderBy("prediction").show()

In [ ]:
# Características de cada cluster
df_clustered.groupBy("prediction").agg(
    avg("anio").alias("anio_promedio"),
    avg("n_ratings").alias("ratings_promedio"),
    avg("rating_promedio").alias("rating_promedio"),
    count("*").alias("peliculas")
).orderBy("prediction").show(truncate=False)

**¿Cómo elegir K?** El método del codo (elbow method): probar varios K y ver el "WSSSE" (suma de distancias al centroide). Donde se "doble" la curva está el K óptimo.

In [ ]:
# Método del codo
import time

resultados = []
for k in [2, 3, 4, 5, 6, 8]:
    km = KMeans(featuresCol="features", k=k, seed=42)
    m = km.fit(df_para_km)
    cost = m.summary.trainingCost
    resultados.append((k, cost))
    print(f"k={k}: WSSSE = {cost:.2f}")

#### Explicación del Código y Resultados del "Método del Codo"
El **Método del Codo (Elbow Method)** es una heurística común utilizada para estimar el número óptimo de clústeres `k` para algoritmos de clustering como K-Means. La idea es ejecutar K-Means para un rango de valores de `k` y calcular una métrica que mida la cohesión de los clústeres. La métrica utilizada aquí es el **WSSSE (Within Set Sum of Squared Errors)** o **Costo de Entrenamiento (`trainingCost`)**.

El WSSSE mide la suma de las distancias al cuadrado de cada punto a su centroide asignado. Un WSSSE más bajo significa clústeres más compactos y mejor definidos. A medida que `k` aumenta, el WSSSE siempre disminuirá (porque los puntos estarán más cerca de sus centroides si hay más centroides para elegir).

El "codo" se refiere al punto en un gráfico de WSSSE vs. K donde la disminución del WSSSE comienza a ralentizarse significativamente, formando una especie de "codo". Este punto es a menudo una buena estimación para el `k` óptimo.

**Explicación del Código:**

```python
import time

resultados = []
for k in [2, 3, 4, 5, 6, 8]:
    km = KMeans(featuresCol="features", k=k, seed=42)
    m = km.fit(df_para_km)
    cost = m.summary.trainingCost
    resultados.append((k, cost))
    print(f"k={k}: WSSSE = {cost:.2f}")
```

1.  **`resultados = []`**: Se inicializa una lista vacía para almacenar los pares `(k, WSSSE)`.
2.  **`for k in [2, 3, 4, 5, 6, 8]:`**: El código itera a través de diferentes valores de `k` (número de clústeres) que se desean probar.
3.  **`km = KMeans(featuresCol="features", k=k, seed=42)`**: Se inicializa un nuevo modelo `KMeans` para cada valor de `k`, utilizando las `features` escaladas.
4.  **`m = km.fit(df_para_km)`**: El modelo `KMeans` se entrena (`fit`) con los datos preparados (`df_para_km`).
5.  **`cost = m.summary.trainingCost`**: Después de entrenar el modelo, se extrae el WSSSE (`trainingCost`) del modelo `m`.
6.  **`resultados.append((k, cost))`**: El valor de `k` y su WSSSE correspondiente se añaden a la lista `resultados`.
7.  **`print(f"k={k}: WSSSE = {cost:.2f}")`**: Se imprime el `k` y el WSSSE para cada iteración.

**Interpretación de los Resultados (`WSSSE`):**

```
k=2: WSSSE = 1203.64
k=3: WSSSE = 632.04
k=4: WSSSE = 481.75
k=5: WSSSE = 438.66
k=6: WSSSE = 290.11
k=8: WSSSE = 230.23
```

Observando los valores de WSSSE:

*   De `k=2` a `k=3`: El WSSSE baja de `1203.64` a `632.04`. Hay una reducción muy grande (`~570`).
*   De `k=3` a `k=4`: El WSSSE baja de `632.04` a `481.75`. Sigue siendo una reducción considerable (`~150`).
*   De `k=4` a `k=5`: El WSSSE baja de `481.75` a `438.66`. La reducción es menor (`~43`).
*   De `k=5` a `k=6`: El WSSSE baja de `438.66` a `290.11`. Hay otra reducción sustancial (`~148`).
*   De `k=6` a `k=8`: El WSSSE baja de `290.11` a `230.23`. La reducción es menor (`~60`).

Si bien la disminución es continua, podemos buscar el punto donde la curva de disminución se "aplanaría" en un gráfico. Parece haber un punto de inflexión o "codo" alrededor de `k=4` o `k=5`, donde la ganancia en reducir el WSSSE por cada `k` adicional empieza a ser menos dramática. También `k=6` parece ser un buen candidato, ya que vuelve a tener una reducción considerable.

La elección final de `k` a menudo es un balance entre un WSSSE bajo y una interpretabilidad de los clústeres. Por ejemplo, `k=4` podría ser una buena opción porque reduce significativamente el WSSSE respecto a `k=3`, y la reducción posterior a `k=5` es menos pronunciada. La elección de `k=6` también podría ser argumentada ya que tiene una buena reducción y los clusters de este k podrían tener un sentido en el contexto del problema.

## 3.2 — Sistema de recomendación con ALS (la corona del módulo)

**ALS** (Alternating Least Squares) es el algoritmo clásico de recomendación basado en **collaborative filtering**.

**Idea:** mirar la matriz `(usuario, película) → rating`. La mayoría está vacía (un usuario ve unas cientos de películas, no millones). ALS aprende dos matrices factorizadas (`U` × `V`) cuya multiplicación reconstruye los ratings observados y **predice los faltantes**.

Es exactamente lo que hacen Netflix, Spotify, Amazon. Y está en MLlib.


In [ ]:
from pyspark.ml.recommendation import ALS

# Train/test split de los ratings
train_als, test_als = df_ratings.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_als.count():,}")
print(f"Test:  {test_als.count():,}")

In [ ]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=20,                  # dimensiones de los vectores latentes
    maxIter=10,
    regParam=0.1,  #  “Modelo, aprende los patrones, pero no te obsesiones demasiado con cada dato exacto.”
    coldStartStrategy="drop", # qué hacer con usuarios/items sin datos
    seed=42
)

print("Entrenando ALS (puede tardar 1-2 min)...")
als_modelo = als.fit(train_als)
print("Listo.")

In [ ]:
# Predicciones sobre test
pred_als = als_modelo.transform(test_als)
pred_als.show(10)

eval_als = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="rmse")
rmse_als = eval_als.evaluate(pred_als)
print(f"RMSE ALS sobre test: {rmse_als:.4f}")

## 3.3 — Generar recomendaciones top-N por usuario

ALS tiene métodos directos: `recommendForAllUsers(N)` da las N mejores películas para cada usuario.


In [ ]:
# Top 5 recomendaciones para cada usuario (limitamos a 10 usuarios para mostrar)
algunos_users = df_ratings.select("userId").distinct().limit(10)

recs = als_modelo.recommendForUserSubset(algunos_users, 5)
recs.show(truncate=False)

In [ ]:
# Hacer las recomendaciones legibles (cruzar con títulos)
from pyspark.sql.functions import explode

recs_legibles = (
    recs
    .withColumn("rec", explode("recommendations"))
    .select(
        "userId",
        col("rec.movieId").alias("movieId"),
        col("rec.rating").alias("score_predicho")
    )
    .join(df_movies.select("movieId", "title"), on="movieId", how="inner")
    .orderBy("userId", col("score_predicho").desc())
)

recs_legibles.show(50, truncate=False)

## 3.4 — Cold start: ¿qué hago con usuarios nuevos?

ALS solo puede recomendar a **usuarios y películas que vio durante el entrenamiento**. Si llega un usuario nuevo, ¿qué hacés?

**Soluciones típicas:**

1. **Onboarding**: pedirle que califique 5-10 películas al registrarse.
2. **Recomendar lo más popular**: top películas por número de ratings.
3. **Content-based**: usar atributos de las películas (género, año) en vez de ratings.
4. **Híbrido**: mezclar ALS con un modelo content-based.

En esta demo, el `coldStartStrategy="drop"` simplemente descarta predicciones para usuarios nuevos.


## 3.5 — Caso integrador: recomendador completo

Ponemos todo junto: dado un usuario, le mostramos:
- Sus 5 películas mejor calificadas (historial).
- Las 5 películas que el modelo le recomienda que no haya visto.


In [ ]:
def recomendar_para_usuario(user_id, top_n=5):
    # 1) Historial del usuario
    historial = (
        df_ratings.filter(col("userId") == user_id)
        .join(df_movies, on="movieId", how="inner")
        .select("title", "rating")
        .orderBy(col("rating").desc())
        .limit(top_n)
    )

    # 2) Recomendaciones para él
    user_df = spark.createDataFrame([(user_id,)], ["userId"])
    recs_user = als_modelo.recommendForUserSubset(user_df, top_n * 3)  # margen para filtrar las vistas
    vistas_ids = [r["movieId"] for r in df_ratings.filter(col("userId") == user_id).collect()]
    recs_explotadas = (
        recs_user
        .withColumn("rec", explode("recommendations"))
        .select(col("rec.movieId").alias("movieId"), col("rec.rating").alias("score"))
        .filter(~col("movieId").isin(vistas_ids))
        .join(df_movies, on="movieId", how="inner")
        .orderBy(col("score").desc())
        .limit(top_n)
    )

    print(f"=== Usuario {user_id} ===")
    print("Sus películas mejor calificadas:")
    historial.show(truncate=False)
    print("Recomendaciones (que aún no vio):")
    recs_explotadas.show(truncate=False)

recomendar_para_usuario(user_id=1)

In [ ]:
recomendar_para_usuario(user_id=42)

## 3.6 — Cierre del módulo

**Lo que se llevan de esta sesión:**

1. **Transformers vs Estimators**: el patrón fundamental de MLlib.
2. **Pipelines**: encadenar preprocesamiento + modelo en una sola unidad.
3. **VectorAssembler / StringIndexer / OneHotEncoder / StandardScaler**: preparar features correctamente.
4. **Regresión lineal**, **regresión logística**, **Random Forest** con sus respectivas métricas.
5. **CrossValidator** para encontrar buenos hiperparámetros sin overfittear.
6. **K-Means** para clustering, y método del codo para elegir K.
7. **ALS** para sistemas de recomendación (la cosa más útil del bloque).
8. **Cold start**: limitaciones reales de los sistemas de recomendación.

---

## Cierre del módulo completo

Llevamos 6 sesiones:

1. **Big Data — Historia y conceptos** (Clase 1, teórica).
2. **DataFrames y transformaciones** (Clase 2, hands-on).
3. **Data Sources y transformaciones avanzadas** (Clase 3, NYC Taxi).
4. **Structured Streaming** (Clase 4, file source + watermarks).
5. **Mejores prácticas y SparkUI** (Clase 5, optimización).
6. **MLlib y sistemas de recomendación** (Clase 6, hoy).

**Lo que pueden hacer ahora con confianza:**
- Tomar un dataset de cualquier tamaño y formato y construir un pipeline batch.
- Optimizarlo con `.explain()`, broadcast, particionado.
- Cambiar a streaming si los datos llegan en vivo.
- Entrenar un modelo ML básico sobre los datos procesados.

**Lo que sigue (no en este módulo, pero conocen el camino):**
- **Delta Lake / Apache Iceberg**: formato lakehouse para versionar datos.
- **Databricks / EMR / Synapse**: correr Spark en producción real.
- **Spark sobre Kubernetes**: para autonomía total.
- **MLflow**: tracking de experimentos ML.

**Tarea final del módulo:** lo que les dejé en la tarea práctica + el bonus de ALS sobre MovieLens (recomendar películas a tu propio perfil después de calificar 10 películas).

Felicitaciones por llegar al final. Cualquier consulta, mi mail está en la portada de las tareas.


In [ ]:
spark.stop()
print("SparkSession cerrada. Fin del módulo.")